## Importing Libraries and Created SparkSession

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder\
        .appName("Bussiness Pipeline & Analytics")\
        .getOrCreate()

## Reading data files

In [0]:
customers_df = spark.read.csv("/Volumes/workspace/default/databrick2027/customers.csv",header = "true",inferSchema = "true")

sales_df = spark.read.csv("/Volumes/workspace/default/databrick2027/sales.csv", header = "true", inferSchema = "true")

customers_df.display()

sales_df.display()

customer_id,customer_name,city
101.0,Aarav,Hyderabad
102.0,Bhavya,Bangalore
103.0,Charan,Chennai
104.0,Divya,Mumbai
105.0,Esha,Pune
106.0,Farhan,Delhi
107.0,Gopal,Hyderabad
108.0,Harsha,Chennai
109.0,Ishita,Bangalore
110.0,John,Mumbai


order_id,customer_id,order_date,product,amount
1001,101,2025-01-18,Keyboard,38303
1002,101,2025-01-18,Mobile,17716
1003,102,2025-01-19,Mobile,33468
1004,102,2025-01-19,Desk,31949
1005,103,2025-01-20,Chair,52689
1006,103,2025-01-20,Mouse,7151
1007,104,2025-01-21,Desk,2857
1008,104,2025-01-21,Chair,29361
1009,105,2025-01-22,Laptop,46602
1010,105,2025-01-22,Desk,18454


##Inspecting data

In [0]:
customers_df.printSchema()

sales_df.printSchema()

root
 |-- customer_id: double (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col

customers_df = customers_df.withColumn(
    "customer_id",
    col("customer_id").cast("int")
)

In [0]:
customers_df.select([count(when(col(c).isNull(), c)).alias(c) for c in customers_df.columns]).show()

sales_df.select([count(when(col(c).isNull(), c)).alias(c) for c in sales_df.columns]).show()

+-----------+-------------+----+
|customer_id|customer_name|city|
+-----------+-------------+----+
|          2|            0|   2|
+-----------+-------------+----+

+--------+-----------+----------+-------+------+
|order_id|customer_id|order_date|product|amount|
+--------+-----------+----------+-------+------+
|       0|          2|         0|      0|     0|
+--------+-----------+----------+-------+------+



## Data Cleaning

### 1.Droping duplicates

In [0]:
customers_df = customers_df.dropDuplicates()

sales_df = sales_df.dropDuplicates()

customers_df.display()
sales_df.display()

customer_id,customer_name,city
101,Aarav,Hyderabad
103,Charan,Chennai
106,Farhan,Delhi
113,Meena,Hyderabad
116,Rahul,Mumbai
129,Rohan,null
null,Unknown,Hyderabad
128,Priya,Hyderabad
109,Ishita,Bangalore
115,Pooja,Bangalore


order_id,customer_id,order_date,product,amount
1042,122,2025-01-11,Chair,25282
1051,null,2025-01-15,Tablet,22000
1017,109,2025-01-26,Mouse,28663
1034,117,2025-01-06,Laptop,32472
1005,103,2025-01-20,Chair,52689
1024,112,2025-01-01,Desk,19991
1005,103,2025-01-20,Keyboard,2500
1011,106,2025-01-23,Mouse,39741
1049,129,2025-01-18,Mouse,36364
1021,111,2025-01-28,Mouse,23655


### 2.Droping null rows using dropna()

In [0]:
customers_df = customers_df.dropna(subset=["customer_id"])

sales_df = sales_df.dropna(subset=["customer_id"])

customers_df.display()
sales_df.display()

customer_id,customer_name,city
101,Aarav,Hyderabad
103,Charan,Chennai
106,Farhan,Delhi
113,Meena,Hyderabad
116,Rahul,Mumbai
129,Rohan,null
128,Priya,Hyderabad
109,Ishita,Bangalore
115,Pooja,Bangalore
125,Manoj,Pune


order_id,customer_id,order_date,product,amount
1042,122,2025-01-11,Chair,25282
1017,109,2025-01-26,Mouse,28663
1034,117,2025-01-06,Laptop,32472
1005,103,2025-01-20,Chair,52689
1024,112,2025-01-01,Desk,19991
1005,103,2025-01-20,Keyboard,2500
1011,106,2025-01-23,Mouse,39741
1049,129,2025-01-18,Mouse,36364
1021,111,2025-01-28,Mouse,23655
1037,119,2025-01-08,Keyboard,25059


### 3. Filling missing city values

In [0]:
customers_df = customers_df.fillna({"City" : "Unknown"})

customers_df.display()

customer_id,customer_name,city
101,Aarav,Hyderabad
103,Charan,Chennai
106,Farhan,Delhi
113,Meena,Hyderabad
116,Rahul,Mumbai
129,Rohan,Unknown
128,Priya,Hyderabad
109,Ishita,Bangalore
115,Pooja,Bangalore
125,Manoj,Pune


## 4. Removing Invalid Sales

In [0]:
sales_df = sales_df.filter(col("amount")>0)
sales_df.display()

order_id,customer_id,order_date,product,amount
1042,122,2025-01-11,Chair,25282
1017,109,2025-01-26,Mouse,28663
1034,117,2025-01-06,Laptop,32472
1005,103,2025-01-20,Chair,52689
1024,112,2025-01-01,Desk,19991
1005,103,2025-01-20,Keyboard,2500
1011,106,2025-01-23,Mouse,39741
1049,129,2025-01-18,Mouse,36364
1021,111,2025-01-28,Mouse,23655
1037,119,2025-01-08,Keyboard,25059


### 5. Coverting date

In [0]:
sales_df = sales_df.withColumn("order_date", to_date("order_date"))

sales_df.display()

order_id,customer_id,order_date,product,amount
1042,122,2025-01-11,Chair,25282
1017,109,2025-01-26,Mouse,28663
1034,117,2025-01-06,Laptop,32472
1005,103,2025-01-20,Chair,52689
1024,112,2025-01-01,Desk,19991
1005,103,2025-01-20,Keyboard,2500
1011,106,2025-01-23,Mouse,39741
1049,129,2025-01-18,Mouse,36364
1021,111,2025-01-28,Mouse,23655
1037,119,2025-01-08,Keyboard,25059


# Tasks

##1. Daily sales

In [0]:
daily_sales = sales_df.groupBy("order_date")\
        .agg(sum("amount").alias("total_sales"))\
        .orderBy(asc("order_date"))

daily_sales.display()

order_date,total_sales
2025-01-01,70860
2025-01-02,41457
2025-01-03,68614
2025-01-04,55154
2025-01-05,95963
2025-01-06,72072
2025-01-07,77893
2025-01-08,31725
2025-01-09,96516
2025-01-10,35140


## 2.City wise revenue

In [0]:
citywise_revenue = customers_df.join(sales_df, on = "customer_id",how = "inner")\
            .groupBy("city")\
            .agg(sum("amount").alias("revenue"))\
            .orderBy("revenue")

citywise_revenue.display()

city,revenue
Unknown,85386
Delhi,207541
Pune,233180
Hyderabad,242666
Mumbai,244480
Bangalore,247458
Chennai,344259


## 3. Top 5 Customers

In [0]:
Top_customers = customers_df.join(sales_df, on = "customer_id", how = "inner")\
                            .groupBy("customer_id","customer_name")\
                            .agg(sum("amount").alias("total_amount"))\
                            .orderBy(desc("total_amount"))\
                            .limit(5)

Top_customers.display()

customer_id,customer_name,total_amount
108,Harsha,104741
120,Yash,96516
116,Rahul,95963
110,John,84542
118,Teja,77893


## 4. Repeat Customers

In [0]:
repeated = sales_df.groupBy("customer_id")\
                        .agg(count("order_id").alias("order_count"))\
                        .filter("order_count>1")\
                        .orderBy(desc("order_count"))

repeated.display()

customer_id,order_count
103,3
109,2
117,2
112,2
106,2
111,2
119,2
120,2
114,2
104,2


## 5. Customer Segmentation

In [0]:
customer_segment = customers_df.join(sales_df,"customer_id")\
                                .groupBy("customer_id","customer_name","city")\
                                .agg(sum("amount").alias("total_spend"),
                                    count("order_id").alias("order_count"))

customer_segment = customer_segment.withColumn(
    "segment",
    when(col("total_spend") >= 90000,"Gold")
    .when(col("total_spend") >= 50000,"Silver")
    .otherwise("Bronze")
)

customer_segment.display()

customer_id,customer_name,city,total_spend,order_count,segment
101,Aarav,Hyderabad,56019,2,Silver
103,Charan,Chennai,62340,3,Silver
106,Farhan,Delhi,61544,2,Silver
113,Meena,Hyderabad,41457,2,Bronze
116,Rahul,Mumbai,95963,2,Gold
129,Rohan,Unknown,36364,1,Bronze
128,Priya,Hyderabad,1806,1,Bronze
109,Ishita,Bangalore,64241,2,Silver
115,Pooja,Bangalore,55154,2,Silver
125,Manoj,Pune,21219,1,Bronze


## 6. Final Reporting Table

In [0]:
final_report = customer_segment.orderBy(desc("total_spend"))

final_report.display()

customer_id,customer_name,city,total_spend,order_count,segment
108,Harsha,Chennai,104741,2,Gold
120,Yash,Chennai,96516,2,Gold
116,Rahul,Mumbai,95963,2,Gold
110,John,Mumbai,84542,2,Silver
118,Teja,Pune,77893,2,Silver
117,Sneha,Hyderabad,72072,2,Silver
112,Lokesh,Delhi,70860,2,Silver
111,Keerthi,Pune,69012,2,Silver
114,Nikhil,Chennai,68614,2,Silver
102,Bhavya,Bangalore,65417,2,Silver


#Load

In [0]:
final_report.write \
    .mode("overwrite") \
    .option("header",True) \
    .csv("/Volumes/workspace/default/databrick2027/final_report")